# PyTorch 网络层详解 - 无参数层

本 Notebook 是系列教程的第3部分，详细介绍 PyTorch 中所有**无参数层**（即内部没有可训练参数的层）。

**主要内容**：
- 激活函数层（Sigmoid、Tanh、ReLU、LeakyReLU、ELU、SELU、GELU、SiLU、Mish、Softplus）
- 池化层（MaxPool2d、AvgPool2d、AdaptiveMaxPool2d、AdaptiveAvgPool2d、MaxUnpool2d、LPPool2d、FractionalMaxPool2d）
- Dropout 层（Dropout、Dropout2d、Dropout3d、AlphaDropout）

In [ ]:
# ============================================================
# 0. 导入必要的库
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

plt.rcParams["font.sans-serif"] = ["SimHei", "WenQuanYi Micro Hei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

print(f"PyTorch version: {torch.__version__}")

## 1. 激活函数层（Activation Layers）

激活函数是神经网络中引入非线性的核心组件。PyTorch 的 `torch.nn` 模块提供了多种激活函数层，它们都继承自 `nn.Module`，可以像其他层一样放入 `nn.Sequential` 中。

**PyTorch 激活函数层速查表**：

| 层（Layer） | 函数式接口 | 公式 | 特点 |
|-------------|-----------|------|------|
| `nn.Sigmoid` | `torch.sigmoid` | $\sigma(x)=1/(1+e^{-x})$ | 输出 (0,1)，饱和 |
| `nn.Tanh` | `torch.tanh` | $\tanh(x)=(e^x-e^{-x})/(e^x+e^{-x})$ | 输出 (-1,1)，零中心 |
| `nn.ReLU` | `F.relu` | $\max(0,x)$ | 计算快，负区死亡 |
| `nn.LeakyReLU` | `F.leaky_relu` | $\max(\alpha x,x)$ | 负区有斜率 |
| `nn.ELU` | `F.elu` | $x$ if $x>0$ else $\alpha(e^x-1)$ | 负区平滑 |
| `nn.SELU` | `F.selu` | $\lambda\cdot\text{ELU}$ | 自归一化 |
| `nn.GELU` | `F.gelu` | $x\cdot\Phi(x)$ | Transformer 标配 |
| `nn.SiLU` | `F.silu` | $x\cdot\sigma(x)$ | Swish，平滑非单调 |
| `nn.Mish` | `F.mish` | $x\cdot\tanh(\text{softplus}(x))$ | YOLOv4 使用 |
| `nn.Softplus` | `F.softplus` | $\ln(1+e^x)$ | ReLU 平滑近似 |

> **注意**：`nn.PReLU` 有可学习参数，属于有参数层，在后续章节介绍。

### 1.1 激活函数层代码示例

以下代码演示如何使用这些激活函数层：

In [ ]:
# ============================================================
# 1.1 激活函数层定义与使用
# ============================================================

# 激活函数层字典：键为层名称，值为 (nn.Module实例, 函数式调用)
activation_layers = {
    'nn.Sigmoid': (nn.Sigmoid(), torch.sigmoid),
    'nn.Tanh': (nn.Tanh(), torch.tanh),
    'nn.ReLU': (nn.ReLU(), F.relu),
    'nn.LeakyReLU': (nn.LeakyReLU(0.1), lambda x: F.leaky_relu(x, 0.1)),
    'nn.ELU': (nn.ELU(alpha=1.0), lambda x: F.elu(x, alpha=1.0)),
    'nn.SELU': (nn.SELU(), F.selu),
    'nn.GELU': (nn.GELU(), F.gelu),
    'nn.SiLU': (nn.SiLU(), F.silu),
    'nn.Mish': (nn.Mish(), F.mish),
    'nn.Softplus': (nn.Softplus(), F.softplus),
}

# 使用示例：创建一个包含激活函数层的 Sequential 模型
model = nn.Sequential(
    nn.Linear(128, 256),
    nn.ReLU(),           # 使用 nn.ReLU 层
    nn.Linear(256, 64),
    nn.GELU(),           # 使用 nn.GELU 层
    nn.Linear(64, 10),
    nn.Sigmoid()         # 输出层使用 Sigmoid
)

print("Sequential 模型结构:")
print(model)
print(f"\n共 {len(activation_layers)} 种激活函数层可供选择")

### 1.2 激活函数曲线与梯度可视化

通过可视化可以直观理解不同激活函数的饱和区（Gradient Saturation）和死亡区（Dead Neuron / Dying ReLU）。

In [ ]:
# ============================================================
# 1.2 绘制激活函数曲线与梯度曲线
# ============================================================

x_vals = torch.linspace(-5, 5, 1000, requires_grad=True)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, (name, (layer, func)) in enumerate(activation_layers.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]
    
    with torch.no_grad():
        y_vals = func(x_vals).detach().numpy()
    
    x_grad = x_vals.clone().detach().requires_grad_(True)
    y_grad = func(x_grad)
    grad = torch.autograd.grad(y_grad.sum(), x_grad)[0].detach().numpy()
    x_np = x_vals.detach().numpy()
    
    ax.plot(x_np, y_vals, 'b-', linewidth=2.5, label=name)
    ax.plot(x_np, grad, 'r--', linewidth=1.5, label='Gradient', alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.5, alpha=0.3)
    ax.axvline(0, color='black', linewidth=0.5, alpha=0.3)
    ax.set_title(name, fontsize=10)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-2, 4)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=8)

for idx in range(len(activation_layers), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('激活函数（实线）与导函数（虚线）', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.3 梯度问题分析

**梯度饱和（Gradient Saturation）**：Sigmoid/Tanh 在两端导数趋近 0，导致梯度消失。

**梯度死亡（Dead Neuron / Dying ReLU）**：ReLU 在负区导数为 0，神经元永久失效。

**梯度消失（Gradient Vanishing）**：深层网络中多个 <1 的梯度连乘导致浅层无法更新。

**梯度爆炸（Gradient Explosion）**：深层网络中多个 >1 的梯度连乘导致梯度剧增。

In [ ]:
# ============================================================
# 1.3 梯度问题分析
# ============================================================

print("=" * 80)
print("梯度问题分析")
print("=" * 80)

def analyze_gradient(func, threshold=0.01):
    x = torch.linspace(-5, 5, 10000, requires_grad=True)
    y = func(x)
    grad = torch.autograd.grad(y.sum(), x)[0].detach().numpy()
    saturated_ratio = np.mean(np.abs(grad) < threshold)
    zero_ratio = np.mean(grad == 0)
    return saturated_ratio, zero_ratio

print(f"{'层名称':<18} {'饱和占比':<14} {'死亡占比':<12} {'备注'}")
print("-" * 80)

for name, (layer, func) in activation_layers.items():
    sat, zero = analyze_gradient(func)
    if zero > 0.01:
        note = "有死亡区 (ReLU负区)"
    elif sat > 0.3:
        note = "有饱和区 (S型函数)"
    else:
        note = "无明显问题"
    print(f"{name:<18} {sat*100:>6.2f}%     {zero*100:>6.2f}%     {note}")

### 1.4 SELU 与自归一化

`nn.SELU` 是 PyTorch 提供的特殊激活函数，具有"自归一化（Self-Normalization）"特性——每层输出自动保持均值为 0、方差为 1，无需 BatchNorm。

**使用条件**（缺一不可）：
1. 激活函数：`nn.SELU()`
2. Dropout：`nn.AlphaDropout()`（不能用普通 Dropout）
3. 权重初始化：`nn.init.lecun_normal_()`
4. 输入数据：标准化（均值 0，方差 1）
5. 网络结构：规整 MLP（不适合 CNN/Transformer）

In [ ]:
# ============================================================
# 1.4 SELU 自归一化验证
# ============================================================

print("=" * 80)
print("SELU 自归一化验证")
print("=" * 80)

def lecun_normal_init(weight):
    fan_in = weight.size(1)
    std = (1.0 / fan_in) ** 0.5
    init.normal_(weight, mean=0.0, std=std)

def test_self_normalization(activation, layers=20, use_lecun=False):
    x = torch.randn(10000, 100)
    linear = nn.Linear(100, 100)
    if use_lecun:
        lecun_normal_init(linear.weight)
        init.zeros_(linear.bias)
    for i in range(layers):
        x = linear(x)
        x = activation(x)
    return x.mean().item(), x.var().item()

configs = [
    ("nn.ReLU (默认)", nn.ReLU(), False),
    ("nn.ELU (默认)", nn.ELU(), False),
    ("nn.SELU (默认)", nn.SELU(), False),
    ("nn.SELU+LeCun", nn.SELU(), True),
]

print(f"{'配置':<22} {'最终均值':<14} {'最终方差':<14} {'状态'}")
print("-" * 70)

for name, act, use_lecun in configs:
    mean, var = test_self_normalization(act, use_lecun=use_lecun)
    is_normalized = abs(mean) < 0.1 and abs(var - 1.0) < 0.2
    status = "✅ 自归一化" if is_normalized else "❌ 否"
    print(f"{name:<22} {mean:>+10.4f}    {var:>10.4f}    {status}")

## 2. 池化层（Pooling Layers）

池化层通过下采样减少特征图尺寸，具有降维和增强平移不变性（Translation Invariance）的作用。

**PyTorch 池化层速查表**：

| 层（Layer） | 函数式接口 | 计算方式 | 特点 |
|-------------|-----------|----------|------|
| `nn.MaxPool2d` | `F.max_pool2d` | 窗口内取最大值 | 保留最显著特征 |
| `nn.AvgPool2d` | `F.avg_pool2d` | 窗口内取平均值 | 平滑特征 |
| `nn.AdaptiveMaxPool2d` | `F.adaptive_max_pool2d` | 固定输出尺寸，取最大值 | 统一输出尺寸 |
| `nn.AdaptiveAvgPool2d` | `F.adaptive_avg_pool2d` | 固定输出尺寸，取平均值 | 统一输出尺寸 |
| `nn.MaxUnpool2d` | — | 恢复最大值位置 | 语义分割上采样 |
| `nn.LPPool2d` | `F.lp_pool2d` | Lp 范数 | 介于最大/平均之间 |
| `nn.FractionalMaxPool2d` | `F.fractional_max_pool2d` | 随机窗口 | 正则化效果 |

**全局池化**：使用 `AdaptiveAvgPool2d(1)` 将整张特征图池化为 1×1，可替代全连接层。

In [ ]:
# ============================================================
# 2.1 准备测试数据
# ============================================================

x = torch.tensor([[
    [1, 2, 3, 4, 5, 6],
    [7, 8, 9, 10, 11, 12],
    [13, 14, 15, 16, 17, 18],
    [19, 20, 21, 22, 23, 24],
    [25, 26, 27, 28, 29, 30],
    [31, 32, 33, 34, 35, 36]
]]).float().unsqueeze(0)  # [1, 1, 6, 6]

print("=" * 80)
print("池化层演示 - 输入数据 (6×6)")
print("=" * 80)
print(x.squeeze().numpy())
print(f"输入形状: {x.shape}\n")

In [ ]:
# ============================================================
# 2.2 MaxPool2d - 最大池化
# ============================================================

print("-" * 80)
print("1. nn.MaxPool2d: 取窗口内最大值")
print("-" * 80)

maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
out = maxpool(x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print("输出:\n", out.squeeze().numpy())

In [ ]:
# ============================================================
# 2.3 AvgPool2d - 平均池化
# ============================================================

print("-" * 80)
print("2. nn.AvgPool2d: 取窗口内平均值")
print("-" * 80)

avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
out = avgpool(x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print("输出:\n", out.squeeze().numpy())

In [ ]:
# ============================================================
# 2.4 最小池化（通过 MaxPool 实现）
# ============================================================

print("-" * 80)
print("3. 最小池化（MinPool）: 取窗口内最小值")
print("-" * 80)

# PyTorch 无直接 MinPool，通过 -MaxPool(-x) 实现
out = -nn.MaxPool2d(2, 2)(-x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print("输出:\n", out.squeeze().numpy())

In [ ]:
# ============================================================
# 2.5 LPPool2d - Lp 池化
# ============================================================

print("-" * 80)
print("4. nn.LPPool2d: 计算窗口内 Lp 范数")
print("-" * 80)

x_small = torch.tensor([[[
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
]]]).float()

print("输入 (4×4):")
print(x_small.squeeze().numpy(), "\n")

print(f"{'p 值':<6} {'Lp 池化输出':<30} {'行为'}")
print("-" * 60)

for p in [1, 2, 3, 5, 10]:
    out = nn.LPPool2d(norm_type=p, kernel_size=2, stride=2)(x_small)
    vals = out.squeeze().numpy().flatten().round(2)
    behavior = {1: "接近平均池化", 2: "L2 范数"}.get(p, "偏向最大池化")
    print(f"p={p:<4} {str(vals):<30} {behavior}")

In [ ]:
# ============================================================
# 2.6 AdaptivePool2d - 自适应池化
# ============================================================

print("-" * 80)
print("5. nn.AdaptiveAvgPool2d / nn.AdaptiveMaxPool2d: 固定输出尺寸")
print("-" * 80)

# 普通池化：输出尺寸取决于输入
pool = nn.MaxPool2d(2, 2)
x6 = torch.randn(1, 1, 6, 6)
x8 = torch.randn(1, 1, 8, 8)
print(f"普通池化 (kernel=2, stride=2):")
print(f"  6×6 → {pool(x6).shape[2]}×{pool(x6).shape[3]}")
print(f"  8×8 → {pool(x8).shape[2]}×{pool(x8).shape[3]}")

# 自适应池化：输出尺寸固定
adaptive = nn.AdaptiveMaxPool2d(3)
print(f"\n自适应池化 (output_size=3):")
print(f"  6×6 → {adaptive(x6).shape[2]}×{adaptive(x6).shape[3]}")
print(f"  8×8 → {adaptive(x8).shape[2]}×{adaptive(x8).shape[3]}")

print("\n自适应池化窗口划分 (6×6 → 3×3):")
print("  ┌───────┬───────┬───────┐")
print("  │ 1  2  │ 3  4  │ 5  6  │")
print("  │ 7  8  │ 9 10  │11 12  │")
print("  ├───────┼───────┼───────┤")
print("  │13 14  │15 16  │17 18  │")
print("  │19 20  │21 22  │23 24  │")
print("  ├───────┼───────┼───────┤")
print("  │25 26  │27 28  │29 30  │")
print("  │31 32  │33 34  │35 36  │")
print("  └───────┴───────┴───────┘")

In [ ]:
# ============================================================
# 2.7 GlobalAvgPool2d - 全局平均池化
# ============================================================

print("-" * 80)
print("6. nn.AdaptiveAvgPool2d(1): 全局平均池化")
print("-" * 80)

global_avg = nn.AdaptiveAvgPool2d((1, 1))
out = global_avg(x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print(f"输出值: {out.item():.2f} (整图均值: {x.mean().item():.2f})")

In [ ]:
# ============================================================
# 2.8 FractionalMaxPool2d - 分数最大池化
# ============================================================

print("-" * 80)
print("7. nn.FractionalMaxPool2d: 随机窗口池化（正则化效果）")
print("-" * 80)

frac_pool = nn.FractionalMaxPool2d(kernel_size=2, output_size=3, return_indices=True)
out, indices = frac_pool(x)
print(f"输入: {x.shape} → 输出: {out.shape}")
print("输出:\n", out.squeeze().numpy())

In [ ]:
# ============================================================
# 2.9 MaxUnpool2d - 最大反池化
# ============================================================

print("-" * 80)
print("8. nn.MaxUnpool2d: 恢复池化位置（需记录索引）")
print("-" * 80)

maxpool_idx = nn.MaxPool2d(2, stride=2, return_indices=True)
x_pool, indices = maxpool_idx(x)

unpool = nn.MaxUnpool2d(kernel_size=2, stride=2)
x_unpool = unpool(x_pool, indices)

print(f"输入: {x.shape} → 池化: {x_pool.shape} → 反池化: {x_unpool.shape}")
print("反池化输出（非零位置为池化时最大值位置）:")
print(x_unpool.squeeze().numpy())

## 3. Dropout 层

Dropout 是防止过拟合的正则化技术：训练时以概率 `p` 随机将神经元输出置零，推理时自动缩放保持期望一致。

**PyTorch Dropout 层速查表**：

| 层（Layer） | 丢弃方式 | 用途 |
|-------------|----------|------|
| `nn.Dropout` | 神经元级随机置零 | 全连接层（MLP） |
| `nn.Dropout2d` | 整通道置零 | 卷积层（保留空间结构） |
| `nn.Dropout3d` | 3D 整通道置零 | 3D 卷积 |
| `nn.AlphaDropout` | 置为固定负值（非零） | SELU 网络（保持自归一化） |

**重要**：训练时需调用 `model.train()`，推理时需调用 `model.eval()`。

In [ ]:
# ============================================================
# 3.1 nn.Dropout - 标准 Dropout
# ============================================================

print("=" * 80)
print("3.1 nn.Dropout: 训练时随机置零，推理时保持不变")
print("=" * 80)

x_sample = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])
dropout = nn.Dropout(p=0.5)

print(f"输入: {x_sample.squeeze().tolist()}\n")

dropout.train()
print("【训练模式】执行 3 次:")
for i in range(3):
    out = dropout(x_sample)
    print(f"  第{i+1}次: {out.squeeze().tolist()}")

dropout.eval()
print(f"\n【推理模式】: {dropout(x_sample).squeeze().tolist()}")

In [ ]:
# ============================================================
# 3.2 不同 p 值对比
# ============================================================

print("\n" + "=" * 80)
print("3.2 不同丢弃概率 p 的影响")
print("=" * 80)

x_fixed = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])
print(f"{'p 值':<8} {'缩放因子':<12} {'输出示例'}")
print("-" * 50)

for p in [0.0, 0.2, 0.5, 0.8]:
    d = nn.Dropout(p)
    d.train()
    out = d(x_fixed)
    scale = 1 / (1 - p)
    print(f"p={p:.1f}    {scale:>5.1f}x        {out.squeeze().tolist()}")

In [ ]:
# ============================================================
# 3.3 nn.Dropout2d - 通道级 Dropout
# ============================================================

print("\n" + "=" * 80)
print("3.3 nn.Dropout2d: 整通道随机丢弃")
print("=" * 80)

x_conv = torch.randn(4, 6, 4, 4)  # [batch, channels, h, w]
dropout2d = nn.Dropout2d(p=0.3)
dropout2d.train()
out = dropout2d(x_conv)

print("每个样本的通道丢弃模式 (1=丢弃):")
for b in range(4):
    status = [int((out[b, c] == 0).all().item()) for c in range(6)]
    print(f"  样本{b}: {status} → 丢弃 {sum(status)}/6 通道")

In [ ]:
# ============================================================
# 3.4 nn.Dropout3d - 3D 通道级 Dropout
# ============================================================

print("\n" + "=" * 80)
print("3.4 nn.Dropout3d: 3D 通道级丢弃")
print("=" * 80)

x3d = torch.randn(4, 8, 8, 8, 8)  # [batch, channels, d, h, w]
dropout3d = nn.Dropout3d(p=0.3)
dropout3d.train()
out = dropout3d(x3d)

zero_ratios = [(out[:, c] == 0).float().mean().item() for c in range(8)]
print(f"各通道零占比: {[f'{r*100:.0f}%' for r in zero_ratios]}")

### 3.5 nn.AlphaDropout

`nn.AlphaDropout` 是专门为 SELU 设计的 Dropout 变体。

**为什么需要它？**

SELU 的自归一化依赖于输入的连续分布。普通 Dropout 会随机将部分神经元置为 0，这会在分布中引入大量离散的 0 值，破坏均值和方差的稳定性。

**AlphaDropout 的两步设计**：

AlphaDropout 分为两步，第一步与丢弃概率 p 无关，第二步与 p 有关。

**第1步：与 p 无关——统一替换为 SELU 负区饱和值**

无论丢弃概率 p 是多少，所有被丢弃的神经元都被替换为同一个固定值：

$$
\text{fill} = -\lambda \cdot \alpha \approx -1.8472
$$

这一步与 p 完全无关，只取决于 SELU 的数学常数（$\lambda \approx 1.0507$，$\alpha \approx 1.6733$）。

```
step1 = mask * x + (1 - mask) * fill
# 被丢弃 → -1.8472，保留 → 原值
```

**设计意图**：保持分布的连续性——用 SELU 负区饱和值替代 0，避免在 0 处产生离散尖峰。

**第2步：与 p 有关——线性变换恢复均值和方差**

第二步对 step1 做线性变换，目的是让输出的均值为 0、方差为 1：

$$
\text{out} = a_p \cdot \text{step1} + b_p
$$

其中 a 和 b 是 p 的函数：

$$
a_p = \frac{1}{\sqrt{q + p q \cdot \text{fill}^2}}, \quad b_p = -a_p \cdot p \cdot \text{fill}
$$

其中 $q = 1-p$。

```
denom = q + (fill ** 2) * p * q
a = 1 / sqrt(denom)
b = -a * fill * p
out = a * step1 + b
```

**设计意图**：无论 p 是多少，都能保证输出均值为 0、方差为 1，维持 SELU 的自归一化。

**被丢弃神经元的最终输出值**：

$$
\text{drop\_final} = a_p \cdot \text{fill} \cdot q
$$

**不同 p 值对比**：

| p | a_p | b_p | 被丢弃神经元最终输出 |
|---|-----|-----|---------------------|
| 0.1 | 0.9821 | 0.1814 | -1.6326 |
| 0.5 | 0.8597 | 0.7940 | -0.7940 |
| 0.9 | 0.5377 | 0.8939 | -0.2482 |

可以看出，随着 p 增大（丢弃更多），被丢弃神经元的最终输出向 0 靠近，但仍保持连续分布。

**使用条件**：必须与 `nn.SELU`、`lecun_normal_` 初始化、标准化输入配套使用。

In [ ]:
# ============================================================
# 3.5 nn.AlphaDropout - 原理演示（p=0.5）
# ============================================================

print("\n" + "=" * 80)
print("3.5 nn.AlphaDropout: p=0.5 完整推导")
print("=" * 80)

# SELU 固定常数
lam = 1.0507009873554804
alpha = 1.7580993408473766
fill = -lam * alpha  # -1.847237

print(f"SELU 常数: λ = {lam:.4f}, α = {alpha:.4f}")
print(f"fill = -λ·α = {fill:.6f}\n")

# p=0.5
p = 0.5
q = 1 - p

print("=" * 40)
print("第1步：与 p 无关——统一替换")
print("=" * 40)
print(f"被丢弃神经元统一替换为: fill = {fill:.6f}\n")

print("=" * 40)
print("第2步：与 p 有关——线性变换")
print("=" * 40)

denom = q + (fill ** 2) * p * q
a = 1 / torch.sqrt(torch.tensor(denom))
b = -a * fill * p

print(f"p = {p}, q = {q}")
print(f"denom = q + fill²·p·q = {denom:.6f}")
print(f"a = 1/√denom = {a.item():.6f}")
print(f"b = -a·fill·p = {b.item():.6f}\n")

drop_final = a * fill * q
print(f"被丢弃神经元最终输出值 = a·fill·q = {drop_final.item():.6f}\n")

# 生成数据演示
torch.manual_seed(10)
x = torch.randn(12)
mask = torch.bernoulli(torch.full_like(x, q))

step1 = mask * x + (1 - mask) * fill
out = a * step1 + b

print("逐神经元演示:")
print(f"{'索引':<6} {'状态':<10} {'原始值':<10} {'中间值':<12} {'最终输出'}")
print("-" * 65)

for idx in range(12):
    m = mask[idx].item()
    status = "被Dropout" if m == 0 else "保留"
    orig = x[idx].item()
    mid = step1[idx].item()
    final = out[idx].item()
    print(f"{idx:<6} {status:<10} {orig:>+8.4f}  {mid:>+10.4f}  {final:>+10.4f}")

print("\n" + "=" * 80)
print("关键结论:")
print(f"  ✅ 所有被丢弃神经元（索引1和2）经过两步后，输出全部等于 {drop_final.item():.6f}")
print(f"  ✅ 第1步与 p 无关（统一替换为 fill={fill:.6f}）")
print(f"  ✅ 第2步与 p 有关（p={p} 时 a={a.item():.6f}, b={b.item():.6f}）")
print(f"  ✅ 分布保持连续，SELU 的自归一化不被破坏")

## 4. 总结

### 4.1 无参数层特点

| 特点 | 说明 |
|------|------|
| **无可训练参数** | 内部没有 `nn.Parameter`，不参与梯度更新 |
| **可放入容器** | 可以放入 `nn.Sequential`、`nn.ModuleList` 等 |
| **继承 nn.Module** | 拥有 `train()`、`eval()`、`to()` 等方法 |
| **状态可切换** | Dropout 训练/推理行为不同 |

### 4.2 层选择速查

| 场景 | 推荐层 | 参数建议 |
|------|--------|----------|
| CNN 下采样 | `nn.MaxPool2d` | kernel_size=2, stride=2 |
| 分类网络末端 | `nn.AdaptiveAvgPool2d(1)` | 替代全连接层 |
| 不同尺寸输入 | `nn.AdaptiveMaxPool2d` | 指定 output_size |
| 语义分割上采样 | `nn.MaxUnpool2d` | 需记录索引 |
| MLP 防止过拟合 | `nn.Dropout` | p=0.3~0.5 |
| CNN 防止过拟合 | `nn.Dropout2d` | p=0.1~0.3 |
| SELU 网络 | `nn.AlphaDropout` | p=0.1~0.5 |
| Transformer | `nn.GELU` | — |
| CNN 标准 | `nn.ReLU` | — |

**关键实践建议**：
1. 训练时 `model.train()`，推理时 `model.eval()`（影响 Dropout）
2. SELU 必须配套 AlphaDropout + LeCun 初始化 + 标准化输入
3. Dropout 放在激活函数之后，全连接层之间
4. BN 与 Dropout 通常不叠加使用